In [ ]:
# 셀 실행 시간 자동 기록 - 텍스트 정제/학습처럼 무거운 셀이 실제로 몇 초/몇 분 걸리는지
# 매번 직접 시간을 재지 않아도, 이 셀 하나만 먼저 실행해두면 이후 모든 셀 끝에 자동으로 찍힌다.
import time
from IPython import get_ipython

def _cell_time_start(info):
    get_ipython()._cell_start_time = time.time()

def _cell_time_end(result):
    start = getattr(get_ipython(), '_cell_start_time', None)
    if start is None:
        return
    elapsed = time.time() - start
    if elapsed >= 60:
        print(f'\n[셀 실행 시간: {int(elapsed // 60)}분 {elapsed % 60:.1f}초]')
    else:
        print(f'\n[셀 실행 시간: {elapsed:.2f}초]')

ip = get_ipython()
ip.events.register('pre_run_cell', _cell_time_start)
ip.events.register('post_run_cell', _cell_time_end)

print('셀 실행 시간 기록을 시작합니다 - 이 셀 다음부터 각 셀 끝에 걸린 시간이 표시됩니다.')

# 구간 A. 기본 전처리
데이터 로드 ~ train/val/test 3분할까지. 릿지 노트북과 동일한 원칙(같은 random_state, 같은 이상치 기준)을 쓴다.

In [ ]:
# 전처리 결과 캐싱(1/2: 불러오기) - 데이터 로드~텍스트 정제(+브랜드 판별 모델 학습까지)는 전체 데이터 기준
# 수십 분씩 걸릴 수 있는데, 커널을 새로 켤 때마다 매번 반복하면 아래쪽(모델 구조/epoch 실험)을 한 번
# 바꿔볼 때마다 이 긴 과정을 처음부터 다시 거쳐야 해서 비효율적이다. 그래서 한 번 계산한 결과를 파일로
# 저장해두고, 다음부터는 파일이 있으면 계산을 통째로 건너뛰고 불러오기만 한다.
# [주의] 전처리/판별모델 로직을 고치면 이 캐시가 예전 로직으로 만든 결과라 안 맞을 수 있다 -
# 그럴 땐 CACHE_PATH 파일을 지우고 다시 실행해서 새로 캐싱해야 한다.
import os
import pickle

CACHE_PATH = 'preprocessed_cache.pkl'

if os.path.exists(CACHE_PATH):
    print(f'캐시 파일 발견 ({CACHE_PATH}) - 아래 전처리 셀들을 건너뛰고 바로 불러옵니다.')
    with open(CACHE_PATH, 'rb') as f:
        _cache = pickle.load(f)
    train_df = _cache['train_df']
    val_df = _cache['val_df']
    test_df = _cache['test_df']
    known_brands = _cache['known_brands']
    brand_pattern = _cache['brand_pattern']
    brand_lower_to_original = _cache['brand_lower_to_original']
    known_cats = _cache['known_cats']
    suspects_df = _cache['suspects_df']
    dist_stats = _cache['dist_stats']
    pos_neg_counts = _cache['pos_neg_counts']
    conf_model_state = _cache['conf_model_state']
    conf_context_vocab = _cache['conf_context_vocab']
    conf_brand_vocab = _cache['conf_brand_vocab']
    conf_hparams = _cache['conf_hparams']
    conf_history_df = _cache['conf_history_df']
    conf_val_metrics = _cache['conf_val_metrics']
    before_after_df = _cache['before_after_df']
    USE_CACHE = True
    print(f'불러오기 완료 - train {len(train_df):,} / val {len(val_df):,} / test {len(test_df):,}')
else:
    print('캐시 없음 - 아래 셀들에서 데이터 로드부터 텍스트 정제까지 처음부터 계산합니다 (시간이 꽤 걸립니다).')
    USE_CACHE = False

In [ ]:
# 데이터 불러오기 - 릿지 노트북(ridge중점.ipynb)과 같은 원본 데이터/이상치 기준을 쓴다.
# (import/설정은 캐시 여부와 무관하게 항상 필요해서 아래 조건문 밖에 둔다)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import copy
import random
from collections import Counter
from IPython.display import display

RANDOM_STATE = 42  # 릿지 노트북과 동일 - test_df를 같은 행으로 맞추기 위함
random.seed(RANDOM_STATE)

# 그래프 한글 깨짐(네모박스) 방지
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

if not USE_CACHE:
    mercari_df = pd.read_csv('../../../data/mercari_train.tsv', sep='\t')

    print(f'전처리 전 : {mercari_df.shape}')
    mercari_df = mercari_df[(mercari_df['price'] > 0) & (mercari_df['price'] <= 2000)]
    print(f'가격 이상치(0원 이하 / 2000달러 초과) 제거 후 : {mercari_df.shape}')

    mercari_df.drop(columns='train_id', inplace=True)
    mercari_df = mercari_df.drop_duplicates()
    mercari_df = mercari_df.reset_index(drop=True)
    print(f'train_id 제거 및 중복 제거 후 : {mercari_df.shape}')

    mercari_df['price'] = np.log1p(mercari_df['price'])
else:
    print('캐시를 사용하므로 원본 tsv 로드는 건너뜁니다.')

In [ ]:
# category_name 3단계 분리 (릿지 노트북과 동일 - 구조적 파싱이라 leakage 없음)
if not USE_CACHE:
    def split_category(df, col='category_name'):
        df = df.copy()
        df[col] = df[col].fillna('Unknown/Unknown/Unknown')
        parts = df[col].str.split('/', n=2, expand=True)
        parts.columns = ['cat_main', 'cat_sub', 'cat_sub2']
        parts = parts.fillna('Unknown')
        return pd.concat([df, parts], axis=1)

    mercari_df = split_category(mercari_df)

In [ ]:
# 결과 확인 - 3단계로 잘 쪼개졌는지 표로 확인
if not USE_CACHE:
    display(mercari_df[['category_name', 'cat_main', 'cat_sub', 'cat_sub2']].head())
else:
    print('캐시 사용 중 - 이 확인 단계는 생략합니다.')

In [ ]:
# 3-way 분리: (trainval 80% / test 20%) -> trainval을 다시 (train 70% / val 10%)로 나눈다.
if not USE_CACHE:
    VAL_SIZE = 0.125  # trainval의 12.5% = 전체의 10%  ->  최종 비율 train 70% / val 10% / test 20%

    trainval_df, test_df = train_test_split(mercari_df, test_size=0.2, random_state=RANDOM_STATE)
    train_df, val_df = train_test_split(trainval_df, test_size=VAL_SIZE, random_state=RANDOM_STATE)

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    total = len(mercari_df)
    print(f'train_df : {train_df.shape}  ({len(train_df)/total:.1%})')
    print(f'val_df   : {val_df.shape}  ({len(val_df)/total:.1%})')
    print(f'test_df  : {test_df.shape}  ({len(test_df)/total:.1%})')
else:
    print(f'train_df : {train_df.shape}')
    print(f'val_df   : {val_df.shape}')
    print(f'test_df  : {test_df.shape}')

# 구간 B. 왜 브랜드 판별 모델이 필요한가
단순 정규식 매칭(+TF-IDF 유사도)이 어떤 문제를 갖는지 실제 데이터로 확인한다.
- 흔한 영단어가 브랜드로 오인되는 문제 (매칭배율 스크리닝)
- 후보가 1개뿐이어도 그게 오답(비교 표현 등)일 수 있다는 문제 (후보 개수별 분포)

In [ ]:
# 브랜드 사전 준비 - train에서만 만든다 (leakage 방지). val/test는 이 사전을 적용만 한다.
# (릿지 노트북에서 검증까지 마친 로직 그대로 - 데이터 품질 문제라 모델 종류와 무관)
if not USE_CACHE:
    WINDOW_SIZE = 5

    def build_known_brands(df, col='brand_name'):
        return sorted(df[col].dropna().unique().tolist())

    def build_brand_pattern(known_brands):
        sorted_brands = sorted([b for b in known_brands if len(b) > 0], key=len, reverse=True)
        escaped = [re.escape(b.lower()) for b in sorted_brands]
        pattern = re.compile(r'\b(' + '|'.join(escaped) + r')\b')
        lower_to_original = {b.lower(): b for b in sorted_brands}
        return pattern, lower_to_original

    def find_all_brand_candidates(text, pattern, lower_to_original, window=WINDOW_SIZE):
        text_lower = str(text).lower()
        candidates = {}
        for match in pattern.finditer(text_lower):
            brand = lower_to_original[match.group(1)]
            if brand in candidates:
                continue
            start, end = match.span()
            before = ' '.join(text_lower[:start].split()[-window:])
            after = ' '.join(text_lower[end:].split()[:window])
            candidates[brand] = f'{before} {match.group(1)} {after}'.strip()
        return candidates

    known_brands = build_known_brands(train_df)
    brand_pattern, brand_lower_to_original = build_brand_pattern(known_brands)

    # brand_source 초기화 - 원래 값이 있던 행은 'original', 없는 행은 일단 'unmatched'
    # (실제 매칭/채우기는 구간 D에서 판별 모델로 한다 - 여기선 "정답을 아는 행"을 구분해두는 용도)
    for _df in (train_df, val_df, test_df):
        _df['brand_source'] = np.where(_df['brand_name'].notna(), 'original', 'unmatched')

    print(f'train 기준 알려진 브랜드 수 : {len(known_brands)}')
else:
    print(f'train 기준 알려진 브랜드 수 : {len(known_brands)} (캐시에서 불러옴)')

In [ ]:
# [구간 B, C 공용] 원래 브랜드가 있던(=정답을 아는) 행들을 훑어서 후보를 전부 찾아본다.
# 이 한 번의 스캔 결과로 아래 세 가지를 동시에 만든다:
#   1) 매칭배율 스크리닝용 카운터 (흔한 단어가 후보로 얼마나 자주 등장하는지)
#   2) 후보 개수별 분포 통계 (0개/1개/2개+, 정답 포함 여부)
#   3) 판별 모델 학습 재료 (item_description 기준 후보별 (문맥, 정답여부) 쌍)
# item_description만 학습 재료로 쓰는 이유: 이미 84만 건대라 데이터가 충분하고, name까지 합치면
# 메모리 대비 이득이 크지 않아 범위를 좁혔다. name 쪽은 분포 통계(구간 B 증거용)까지만 확인한다.
if not USE_CACHE:
    original_train_df = train_df[train_df['brand_source'] == 'original']

    candidate_counter = Counter()          # 매칭배율용 - (텍스트 종류 무관) 후보로 등장한 총 횟수
    dist_stats = {
        'item_description': {'no_candidate': 0, 'single': 0, 'single_correct': 0, 'single_wrong': 0,
                              'multi': 0, 'multi_has_true': 0, 'multi_no_true': 0},
        'name': {'no_candidate': 0, 'single': 0, 'single_correct': 0, 'single_wrong': 0,
                 'multi': 0, 'multi_has_true': 0, 'multi_no_true': 0},
    }
    candidate_records = []  # item_description 기준 학습 재료만 여기 쌓는다

    _zipped = zip(
        original_train_df['item_description'], original_train_df['name'], original_train_df['brand_name'],
        original_train_df['cat_main'], original_train_df['cat_sub'], original_train_df['cat_sub2'],
    )
    for desc_text, name_text, true_brand, cat_main, cat_sub, cat_sub2 in _zipped:
        cat_context = f'{cat_main} {cat_sub} {cat_sub2}'
        name_context = str(name_text)
        for col_name, text in (('item_description', desc_text), ('name', name_text)):
            candidates = find_all_brand_candidates(text, brand_pattern, brand_lower_to_original)
            for cand in candidates:
                candidate_counter[cand] += 1

            n_cand = len(candidates)
            stats = dist_stats[col_name]
            if n_cand == 0:
                stats['no_candidate'] += 1
            elif n_cand == 1:
                stats['single'] += 1
                only_brand = next(iter(candidates))
                if only_brand == true_brand:
                    stats['single_correct'] += 1
                else:
                    stats['single_wrong'] += 1
            else:
                stats['multi'] += 1
                if true_brand in candidates:
                    stats['multi_has_true'] += 1
                else:
                    stats['multi_no_true'] += 1

            if col_name == 'item_description':
                for cand_brand, window_text in candidates.items():
                    candidate_records.append({
                        'brand': cand_brand, 'window': window_text,
                        'cat_context': cat_context, 'name_context': name_context,
                        'label': 1 if cand_brand == true_brand else 0,
                    })

    print(f'스캔한 행 수 : {len(original_train_df):,}')
    print(f'item_description 기준 학습 재료(후보 1건당 1개) : {len(candidate_records):,}개')
else:
    print('캐시 사용 중 - 스캔 과정은 생략하고 이미 집계된 결과만 아래에서 확인합니다.')

In [ ]:
# 결과 확인 - 매칭배율 스크리닝 (한 단어짜리 브랜드만, train 기준)
# 매칭배율 = (후보로 등장한 총 횟수) / (원래 그 브랜드로 실제 채워져 있던 행 수 + 1)
# 비율이 비정상적으로 높으면 "흔한 영단어가 브랜드로 오인되고 있다"는 신호다.
if not USE_CACHE:
    _original_counts = original_train_df['brand_name'].value_counts()
    _rows = []
    for _brand, _cand_count in candidate_counter.items():
        if ' ' in _brand or _brand == 'Unknown':
            continue  # 여러 단어로 된 브랜드는 우연히 통째로 매칭될 확률이 낮아 상대적으로 안전
        _orig_count = int(_original_counts.get(_brand, 0))
        _rows.append({'brand_name': _brand, 'original': _orig_count, 'candidate_count': _cand_count,
                       '매칭배율': _cand_count / (_orig_count + 1)})
    suspects_df = pd.DataFrame(_rows).sort_values('매칭배율', ascending=False).reset_index(drop=True)

with pd.option_context('display.float_format', '{:,.1f}'.format):
    display(suspects_df.head(20))

In [ ]:
# 결과 확인 - 후보 개수별 분포 (item_description / name 각각)
# "1개여도 오답일 수 있다"는 걸 숫자로 확인한다.
_rows = []
for _col, _stats in dist_stats.items():
    _total = sum(v for k, v in _stats.items() if k in ('no_candidate', 'single', 'multi'))
    _rows.append({
        '텍스트': _col, '전체 행': _total,
        '후보 0개': _stats['no_candidate'],
        '후보 1개': _stats['single'], '  ↳ 정답': _stats['single_correct'], '  ↳ 오답': _stats['single_wrong'],
        '후보 2개+': _stats['multi'], '  ↳정답포함': _stats['multi_has_true'], '  ↳정답없음': _stats['multi_no_true'],
    })
dist_df = pd.DataFrame(_rows)
with pd.option_context('display.float_format', '{:,.0f}'.format):
    display(dist_df)

# 구간 C. 브랜드 판별 신뢰도 모델 학습
"이 후보 브랜드 + 주변 문맥을 보니, 이게 진짜 이 상품의 브랜드일 확률이 얼마나 되나?"를 배우는 작은 이진분류 모델. 후보가 1개든 여러 개든 항상 같은 방식으로 검증한다 (Word2Vec의 CBOW 발상 응용).

In [ ]:
# 학습 데이터 정리 - 긍정(후보=정답) / 부정(후보≠정답, 비교표현 등 포함) 개수 확인 + 클래스 균형 맞추기
# 부정이 훨씬 많으므로(약 3.5배), 부정을 긍정 개수만큼 무작위로 덜어내서 1:1로 맞춘다.
if not USE_CACHE:
    candidates_df = pd.DataFrame(candidate_records)
    pos_df = candidates_df[candidates_df['label'] == 1]
    neg_df = candidates_df[candidates_df['label'] == 0]

    pos_neg_counts = {'긍정(후보=정답)': len(pos_df), '부정(후보≠정답)': len(neg_df)}

    neg_sampled = neg_df.sample(n=len(pos_df), random_state=RANDOM_STATE)
    balanced_df = pd.concat([pos_df, neg_sampled], ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f'균형 맞추기 후 학습 재료 : {len(balanced_df):,}건 (긍정 {len(pos_df):,} + 부정(샘플) {len(neg_sampled):,})')
else:
    print('캐시 사용 중')

print()
print(pd.Series(pos_neg_counts, name='건수').to_string())

In [ ]:
# 판별 모델 입력 준비 - 문맥(윈도우+카테고리+상품명)을 하나로 합쳐 간단히 토큰화하고,
# 등장 빈도 상위 단어로 작은 어휘사전(context_vocab)을 만든다. 브랜드는 known_brands 전체로 사전을 만든다.
# (이 시점엔 아직 본 파이프라인의 text_cleaning 함수가 안 만들어져 있어서, 여기서는 간단한 소문자+split만 쓴다)
if not USE_CACHE:
    CONTEXT_VOCAB_SIZE = 15000
    CONF_MAXLEN = 30

    def simple_tokenize(text):
        return re.sub(r'[^a-z0-9\s]', ' ', str(text).lower()).split()

    balanced_df['combined_text'] = (
        balanced_df['window'] + ' ' + balanced_df['cat_context'] + ' ' + balanced_df['name_context']
    ).apply(simple_tokenize)

    _counter = Counter()
    for _toks in balanced_df['combined_text']:
        _counter.update(_toks)
    conf_context_vocab = {w: i + 2 for i, (w, _) in enumerate(_counter.most_common(CONTEXT_VOCAB_SIZE))}  # 0=PAD,1=UNK
    conf_brand_vocab = {b: i + 1 for i, b in enumerate(sorted(known_brands))}  # 0=미확인용 예비

    def to_padded_ids(tokens, vocab, maxlen):
        ids = np.zeros(maxlen, dtype=np.int64)
        idx = [vocab.get(t, 1) for t in tokens[:maxlen]]
        ids[:len(idx)] = idx
        return ids

    conf_context_ids = np.stack([to_padded_ids(t, conf_context_vocab, CONF_MAXLEN) for t in balanced_df['combined_text']])
    conf_brand_ids = balanced_df['brand'].map(conf_brand_vocab).fillna(0).astype(np.int64).values
    conf_labels = balanced_df['label'].astype(np.float32).values

    print(f'문맥 어휘 수 : {len(conf_context_vocab):,} (+PAD/UNK)')
    print(f'브랜드 어휘 수 : {len(conf_brand_vocab):,}')
    print(f'입력 shape : context_ids {conf_context_ids.shape}, brand_ids {conf_brand_ids.shape}')

In [ ]:
# 판별 모델 자체의 train/val 분할 (9:1) - 이 모델이 실제로 잘 맞히는지 검증하려면 학습에 안 쓴 데이터가 필요하다.
if not USE_CACHE:
    _n = len(conf_labels)
    _idx = np.arange(_n)
    rng = np.random.RandomState(RANDOM_STATE)
    rng.shuffle(_idx)
    _n_val = int(_n * 0.1)
    conf_val_idx, conf_train_idx = _idx[:_n_val], _idx[_n_val:]

    print(f'판별모델 train : {len(conf_train_idx):,}건,  val : {len(conf_val_idx):,}건')

In [ ]:
# 판별 모델 정의 - 문맥(윈도우+카테고리+상품명 합친 것)을 임베딩+평균, 후보 브랜드도 임베딩,
# 이 둘을 이어붙여 Dense층 통과 후 "진짜 브랜드일 확률"에 대한 로짓(logit) 1개를 출력한다.
class BrandConfidenceModel(nn.Module):
    def __init__(self, context_vocab_size, n_brand, word_dim=32, brand_dim=16, hidden=128, dropout=0.3):
        super().__init__()
        self.context_emb = nn.EmbeddingBag(context_vocab_size + 2, word_dim, mode='mean', padding_idx=0)
        self.brand_emb = nn.Embedding(n_brand + 1, brand_dim, padding_idx=0)
        self.mlp = nn.Sequential(
            nn.Linear(word_dim + brand_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, context_ids, brand_ids):
        x = torch.cat([self.context_emb(context_ids), self.brand_emb(brand_ids)], dim=1)
        return self.mlp(x).squeeze(1)

if not USE_CACHE:
    conf_model = BrandConfidenceModel(len(conf_context_vocab), len(conf_brand_vocab)).to(device)
    conf_hparams = {'context_vocab_size': len(conf_context_vocab), 'n_brand': len(conf_brand_vocab)}
    print(conf_model)
    print(f'파라미터 수 : {sum(p.numel() for p in conf_model.parameters()):,}')

In [ ]:
# 판별 모델 학습 - 이진분류라 BCEWithLogitsLoss 사용. val 정확도 기준 조기종료(메인 모델과 동일한 방식).
if not USE_CACHE:
    def make_conf_batches(idx_array, batch_size, shuffle):
        idx = idx_array.copy()
        if shuffle:
            np.random.RandomState(RANDOM_STATE).shuffle(idx)
        for start in range(0, len(idx), batch_size):
            yield idx[start:start + batch_size]

    def run_conf_epoch(model, idx_array, optimizer=None, batch_size=1024):
        is_train = optimizer is not None
        model.train() if is_train else model.eval()
        criterion = nn.BCEWithLogitsLoss()
        total_loss, n_correct, n_total = 0.0, 0, 0

        with torch.set_grad_enabled(is_train):
            for batch_idx in make_conf_batches(idx_array, batch_size, shuffle=is_train):
                ctx = torch.as_tensor(conf_context_ids[batch_idx], dtype=torch.long, device=device)
                brd = torch.as_tensor(conf_brand_ids[batch_idx], dtype=torch.long, device=device)
                y = torch.as_tensor(conf_labels[batch_idx], dtype=torch.float32, device=device)

                logits = model(ctx, brd)
                loss = criterion(logits, y)

                if is_train:
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                total_loss += loss.item() * len(batch_idx)
                n_correct += ((torch.sigmoid(logits) >= 0.5).float() == y).sum().item()
                n_total += len(batch_idx)

        return total_loss / n_total, n_correct / n_total

    CONF_N_EPOCHS = 8
    CONF_PATIENCE = 2
    conf_optimizer = torch.optim.Adam(conf_model.parameters(), lr=1e-3)

    conf_best_acc = 0.0
    conf_best_state = None
    conf_no_improve = 0
    conf_history = []

    for epoch in range(1, CONF_N_EPOCHS + 1):
        train_loss, train_acc = run_conf_epoch(conf_model, conf_train_idx, conf_optimizer)
        val_loss, val_acc = run_conf_epoch(conf_model, conf_val_idx, optimizer=None)
        conf_history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                              'val_loss': val_loss, 'val_acc': val_acc})
        print(f'epoch {epoch}/{CONF_N_EPOCHS}  train loss={train_loss:.4f} acc={train_acc:.4f}  '
              f'val loss={val_loss:.4f} acc={val_acc:.4f}', end='')

        if val_acc > conf_best_acc:
            conf_best_acc = val_acc
            conf_best_state = copy.deepcopy(conf_model.state_dict())
            conf_no_improve = 0
            print('  <- best')
        else:
            conf_no_improve += 1
            print(f'  (연속 {conf_no_improve}회 개선 없음)')
            if conf_no_improve >= CONF_PATIENCE:
                print(f'\nval 정확도가 {CONF_PATIENCE}epoch 연속 개선되지 않아 조기 종료합니다.')
                break

    conf_model.load_state_dict(conf_best_state)
    conf_model_state = conf_best_state
    conf_history_df = pd.DataFrame(conf_history)
    print(f'\n최적 val 정확도 : {conf_best_acc:.4f}')
else:
    conf_model = BrandConfidenceModel(**conf_hparams).to(device)
    conf_model.load_state_dict(conf_model_state)
    conf_model.eval()
    print('캐시된 판별 모델 가중치를 불러왔습니다.')

In [ ]:
# 결과 확인 - 판별 모델 학습곡선
plt.figure(figsize=(6, 4))
plt.plot(conf_history_df['epoch'], conf_history_df['train_acc'], marker='o', color='#2a78d6', label='train acc')
plt.plot(conf_history_df['epoch'], conf_history_df['val_acc'], marker='o', color='#eb6834', label='val acc')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.title('브랜드 판별 모델 학습곡선')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 결과 확인 - 판별 모델 최종 검증 (정확도/정밀도/재현율)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

if not USE_CACHE:
    conf_model.eval()
    with torch.no_grad():
        ctx = torch.as_tensor(conf_context_ids[conf_val_idx], dtype=torch.long, device=device)
        brd = torch.as_tensor(conf_brand_ids[conf_val_idx], dtype=torch.long, device=device)
        val_probs = torch.sigmoid(conf_model(ctx, brd)).cpu().numpy()
    val_preds = (val_probs >= 0.5).astype(int)
    val_true = conf_labels[conf_val_idx].astype(int)

    acc = accuracy_score(val_true, val_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(val_true, val_preds, average='binary', zero_division=0)
    conf_val_metrics = {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

metrics_df = pd.DataFrame([conf_val_metrics]).rename(index={0: 'val 검증 결과'})
with pd.option_context('display.float_format', '{:.4f}'.format):
    display(metrics_df)

# 구간 D. 판별 모델로 브랜드 결측치 채우기
기존 정규식 매칭은 그대로 쓰되, "이 후보가 진짜 브랜드인가"의 최종 판단만 TF-IDF 유사도 대신 구간 C에서 학습한 판별 모델로 한다. 후보가 1개여도 검증을 거친다.

In [ ]:
# 판별 모델 기반 브랜드 채우기 - 후보가 몇 개든 전부 판별 모델로 점수를 매겨서, 임계값을 넘는 것 중
# 최고점만 채택한다. 하나도 못 넘으면 채우지 않고 그대로 NaN(-> 나중에 Unknown)으로 남긴다.
if not USE_CACHE:
    CONF_THRESHOLD = 0.5

    def score_candidates_with_model(candidates, cat_context, name_context):
        if len(candidates) == 0:
            return {}
        brands = list(candidates.keys())
        combined_texts = [simple_tokenize(f'{candidates[b]} {cat_context} {name_context}') for b in brands]
        ctx_ids = np.stack([to_padded_ids(t, conf_context_vocab, CONF_MAXLEN) for t in combined_texts])
        brd_ids = np.array([conf_brand_vocab.get(b, 0) for b in brands], dtype=np.int64)
        with torch.no_grad():
            ctx_t = torch.as_tensor(ctx_ids, dtype=torch.long, device=device)
            brd_t = torch.as_tensor(brd_ids, dtype=torch.long, device=device)
            probs = torch.sigmoid(conf_model(ctx_t, brd_t)).cpu().numpy()
        return dict(zip(brands, probs))

    def fill_brand_with_model(df, pattern, lower_to_original, desc_col,
                               brand_col='brand_name', source_col='brand_source'):
        df = df.copy()
        missing_mask = df[brand_col].isna()

        def find_brand(row):
            text = row[desc_col]
            text = '' if pd.isna(text) else text
            candidates = find_all_brand_candidates(text, pattern, lower_to_original)
            if len(candidates) == 0:
                return None
            cat_context = ' '.join(str(row[c]) for c in ['cat_main', 'cat_sub', 'cat_sub2'])
            name_context = str(row['name'])
            scores = score_candidates_with_model(candidates, cat_context, name_context)
            best_brand = max(scores, key=scores.get)
            return best_brand if scores[best_brand] >= CONF_THRESHOLD else None

        found = df.loc[missing_mask].apply(find_brand, axis=1)
        df.loc[missing_mask, brand_col] = found
        newly_filled_mask = missing_mask & df[brand_col].notna()
        df.loc[newly_filled_mask, source_col] = desc_col
        return df

    def apply_brand_fill_model(df):
        df = fill_brand_with_model(df, brand_pattern, brand_lower_to_original, desc_col='item_description')
        df = fill_brand_with_model(df, brand_pattern, brand_lower_to_original, desc_col='name')
        return df

    train_df = apply_brand_fill_model(train_df)
    val_df = apply_brand_fill_model(val_df)
    test_df = apply_brand_fill_model(test_df)

    known_brands_set = set(known_brands)

    def restrict_to_known(series, known_set, unknown_value='Unknown'):
        return series.where(series.isin(known_set) & series.notna(), unknown_value)

    train_df['brand_name'] = restrict_to_known(train_df['brand_name'], known_brands_set)
    val_df['brand_name'] = restrict_to_known(val_df['brand_name'], known_brands_set)
    test_df['brand_name'] = restrict_to_known(test_df['brand_name'], known_brands_set)

    known_cats = {}
    for col in ['cat_main', 'cat_sub', 'cat_sub2']:
        known_cats[col] = set(train_df[col].unique())
        val_df[col] = restrict_to_known(val_df[col], known_cats[col])
        test_df[col] = restrict_to_known(test_df[col], known_cats[col])

    print('brand_name == Unknown (train):', (train_df['brand_name'] == 'Unknown').sum())
    print('brand_name == Unknown (val)  :', (val_df['brand_name'] == 'Unknown').sum())
    print('brand_name == Unknown (test) :', (test_df['brand_name'] == 'Unknown').sum())
else:
    print('brand_name == Unknown (train):', (train_df['brand_name'] == 'Unknown').sum())
    print('brand_name == Unknown (val)  :', (val_df['brand_name'] == 'Unknown').sum())
    print('brand_name == Unknown (test) :', (test_df['brand_name'] == 'Unknown').sum())

In [ ]:
# 결과 확인 - Before/After 비교표
# "예전 방식(후보 1개면 무조건 채움)이었다면 어떻게 됐을지" vs "판별 모델의 실제 결정"을 나란히 놓고
# 확인한다. 판별 모델이 낮은 점수를 준 케이스(비교표현 등으로 의심됨)가 실제로 걸러지는지가 핵심 증거다.
if not USE_CACHE:
    _sample_pool = train_df[(train_df['brand_source'] == 'unmatched') | (train_df['brand_source'] == 'item_description')]
    _examples = []
    for _, row in _sample_pool.sample(min(3000, len(_sample_pool)), random_state=RANDOM_STATE).iterrows():
        text = row['item_description']
        candidates = find_all_brand_candidates(text, brand_pattern, brand_lower_to_original)
        if len(candidates) == 0:
            continue
        cat_context = ' '.join(str(row[c]) for c in ['cat_main', 'cat_sub', 'cat_sub2'])
        name_context = str(row['name'])
        scores = score_candidates_with_model(candidates, cat_context, name_context)
        naive_brand = next(iter(candidates))  # 예전 방식: 그냥 처음 찾은 후보를 채택
        best_brand = max(scores, key=scores.get)
        best_score = scores[best_brand]
        new_decision = best_brand if best_score >= CONF_THRESHOLD else 'Unknown(거부)'
        _examples.append({
            '상품명': row['name'], '매칭 근거 텍스트': text[:120],
            '후보': ', '.join(candidates.keys()),
            '예전 방식이면 채택': naive_brand,
            '판별모델 최고점수 후보': f'{best_brand} ({best_score:.2f})',
            '판별모델 최종결정': new_decision,
            '결정 달라짐': 'O' if new_decision != naive_brand else '',
        })
        if len(_examples) >= 400:
            break

    before_after_df = pd.DataFrame(_examples)
    _changed = before_after_df[before_after_df['결정 달라짐'] == 'O']
    print(f'표본 {len(before_after_df)}건 중 판별모델이 예전 방식과 다르게 결정한 건 : {len(_changed)}건')

before_after_df[before_after_df['결정 달라짐'] == 'O'].head(8)

In [ ]:
# 결과 확인 - brand_source 분포
print('train brand_source :'); print(train_df['brand_source'].value_counts())
print('\nval brand_source   :'); print(val_df['brand_source'].value_counts())

# 구간 E. 나머지 전처리
추가 feature 계산, 텍스트 정제(토큰화), 그리고 여기까지 전체를 캐싱.

In [ ]:
# 추가 feature: 설명 유무 플래그 / 텍스트 길이 (릿지 노트북과 동일 - 원문 상태에서 계산)
if not USE_CACHE:
    for df in (train_df, val_df, test_df):
        df['has_description'] = (
            df['item_description'].fillna('No description yet') != 'No description yet'
        ).astype(int)
        df['name_len'] = df['name'].fillna('').apply(lambda x: len(str(x).split()))
        df['desc_len'] = df['item_description'].fillna('No description yet').apply(lambda x: len(str(x).split()))

In [ ]:
# 결과 확인 - 과학적 표기 방지 처리
with pd.option_context('display.float_format', '{:,.2f}'.format):
    display(train_df[['has_description', 'name_len', 'desc_len']].describe())

In [ ]:
# 텍스트 정제 - 불용어 제거는 하지 않는다 (임베딩엔 그 이유가 안 맞아서).
if not USE_CACHE:
    import nltk
    from nltk.tokenize import word_tokenize
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

    def text_cleaning(text):
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    def preprocess_text(text):
        return word_tokenize(text_cleaning(text))

    def clean_text_columns(df):
        df = df.copy()
        df['item_description'] = df['item_description'].fillna('No description yet').astype(str)
        df['name'] = df['name'].fillna('').astype(str)
        df['name_tokens'] = df['name'].apply(preprocess_text)
        df['desc_tokens'] = df['item_description'].apply(preprocess_text)
        return df

    # 데이터 규모가 커서 시간이 다소 걸릴 수 있음 - 여기까지가 캐싱되는 구간이다.
    train_df = clean_text_columns(train_df)
    val_df = clean_text_columns(val_df)
    test_df = clean_text_columns(test_df)

In [ ]:
# 결과 확인 - 정제된 토큰이 잘 만들어졌는지 표로 확인
train_df[['name', 'name_tokens', 'desc_tokens']].head()

In [ ]:
# 전처리 결과 캐싱(2/2: 저장하기) - 방금 새로 계산했을 때만(캐시가 없었을 때만) 다음번을 위해 파일로 저장해둔다.
if not USE_CACHE:
    with open(CACHE_PATH, 'wb') as f:
        pickle.dump({
            'train_df': train_df, 'val_df': val_df, 'test_df': test_df,
            'known_brands': known_brands, 'brand_pattern': brand_pattern,
            'brand_lower_to_original': brand_lower_to_original, 'known_cats': known_cats,
            'suspects_df': suspects_df, 'dist_stats': dist_stats, 'pos_neg_counts': pos_neg_counts,
            'conf_model_state': conf_model_state, 'conf_context_vocab': conf_context_vocab,
            'conf_brand_vocab': conf_brand_vocab, 'conf_hparams': conf_hparams,
            'conf_history_df': conf_history_df, 'conf_val_metrics': conf_val_metrics,
            'before_after_df': before_after_df,
        }, f)
    print(f'전처리 결과를 {CACHE_PATH}에 저장했습니다 - 다음 실행부터는 이 파일에서 바로 불러옵니다.')
else:
    print('캐시에서 불러온 데이터라 다시 저장하지 않습니다.')

# 구간 F. 딥러닝 입력 준비
시퀀스 길이 결정, 어휘사전, OOV 확인, 라벨 인코딩(희귀 버킷), 스케일링.

In [ ]:
# 시퀀스 길이(maxlen) 결정용 - 실제 정제된 토큰 길이를 계산
name_token_len = train_df['name_tokens'].apply(len)
desc_token_len = train_df['desc_tokens'].apply(len)

In [ ]:
# 결과 확인 - name/description 토큰 길이 분포 비교
token_len_summary = pd.DataFrame({
    'name': name_token_len.describe(percentiles=[.5, .75, .9, .95, .99]),
    'description': desc_token_len.describe(percentiles=[.5, .75, .9, .95, .99]),
})
with pd.option_context('display.float_format', '{:,.2f}'.format):
    display(token_len_summary)

In [ ]:
# 위 표의 95백분위수를 기준으로 maxlen을 정한다.
NAME_MAXLEN = int(np.ceil(name_token_len.quantile(0.95)))
DESC_MAXLEN = int(np.ceil(desc_token_len.quantile(0.95)))
print(f'NAME_MAXLEN = {NAME_MAXLEN} (95백분위수 기준)')
print(f'DESC_MAXLEN = {DESC_MAXLEN} (95백분위수 기준)')

In [ ]:
# 어휘 사전(vocab) 구축 - train_df 토큰만 사용(leakage 방지), 등장 빈도 상위 N개만 사용
NAME_VOCAB_SIZE = 30000
DESC_VOCAB_SIZE = 50000

def build_vocab(token_series, vocab_size):
    counter = Counter()
    for tokens in token_series:
        counter.update(tokens)
    return counter, {w: i + 2 for i, (w, _) in enumerate(counter.most_common(vocab_size))}  # 0=PAD, 1=UNK

name_counter, name_vocab = build_vocab(train_df['name_tokens'], NAME_VOCAB_SIZE)
desc_counter, desc_vocab = build_vocab(train_df['desc_tokens'], DESC_VOCAB_SIZE)
print(f'name 어휘 수 : {len(name_vocab)} (+PAD/UNK)')
print(f'description 어휘 수 : {len(desc_vocab)} (+PAD/UNK)')

def tokens_to_padded_ids(token_series, vocab, maxlen):
    ids = np.zeros((len(token_series), maxlen), dtype=np.int64)
    for i, tokens in enumerate(token_series):
        idx = [vocab.get(t, 1) for t in tokens[:maxlen]]
        ids[i, :len(idx)] = idx
    return ids

name_train_ids = tokens_to_padded_ids(train_df['name_tokens'], name_vocab, NAME_MAXLEN)
name_val_ids = tokens_to_padded_ids(val_df['name_tokens'], name_vocab, NAME_MAXLEN)
name_test_ids = tokens_to_padded_ids(test_df['name_tokens'], name_vocab, NAME_MAXLEN)
desc_train_ids = tokens_to_padded_ids(train_df['desc_tokens'], desc_vocab, DESC_MAXLEN)
desc_val_ids = tokens_to_padded_ids(val_df['desc_tokens'], desc_vocab, DESC_MAXLEN)
desc_test_ids = tokens_to_padded_ids(test_df['desc_tokens'], desc_vocab, DESC_MAXLEN)

print('name 시퀀스 shape       :', name_train_ids.shape, name_val_ids.shape, name_test_ids.shape)
print('description 시퀀스 shape:', desc_train_ids.shape, desc_val_ids.shape, desc_test_ids.shape)

In [ ]:
# 결과 확인 - 어휘사전에 실제로 어떤 단어들이 들어갔는지 (상위 빈도 20개)
top_words_df = pd.DataFrame({
    'name 상위 단어': [w for w, _ in name_counter.most_common(20)],
    'name 빈도': [c for _, c in name_counter.most_common(20)],
    'description 상위 단어': [w for w, _ in desc_counter.most_common(20)],
    'description 빈도': [c for _, c in desc_counter.most_common(20)],
})
top_words_df

In [ ]:
# OOV(사전에 없는 단어) 비율 확인
def oov_rate(token_series, vocab):
    total, oov = 0, 0
    for tokens in token_series:
        total += len(tokens)
        oov += sum(1 for t in tokens if t not in vocab)
    return oov / total if total else 0.0

for split_name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    name_oov = oov_rate(df['name_tokens'], name_vocab)
    desc_oov = oov_rate(df['desc_tokens'], desc_vocab)
    print(f'[{split_name:5s}] name OOV 비율 : {name_oov:.2%}   description OOV 비율 : {desc_oov:.2%}')

In [ ]:
# 결과 확인 - 라벨 인코딩(희귀 버킷) 기준을 정하기 전에, 브랜드별 등장 횟수 분포를 먼저 본다.
brand_counts = train_df['brand_name'].value_counts()
plt.figure(figsize=(7, 4))
plt.hist(brand_counts.values, bins=60, color='#2a78d6')
plt.yscale('log')
plt.xlabel('브랜드 하나당 등장 횟수')
plt.ylabel('브랜드 종류 수 (로그 스케일)')
plt.title('브랜드별 등장 횟수 분포 (대부분 왼쪽에 몰려있음 = 희귀 브랜드가 압도적으로 많음)')
plt.axvline(10, color='#eb6834', linestyle='--', label='MIN_CATEGORY_FREQ=10 기준선')
plt.legend()
plt.tight_layout()
plt.show()

print(f'브랜드 총 종류 : {train_df["brand_name"].nunique():,}종')
print(f'그중 등장 10회 미만 : {(brand_counts < 10).sum():,}종 ({(brand_counts < 10).mean():.1%})')

In [ ]:
# 브랜드/카테고리 라벨 인코딩 - 위 분포에서 봤듯 희귀 브랜드가 압도적으로 많아, 10회 미만은 "희귀" 버킷(=0)으로 묶는다.
MIN_CATEGORY_FREQ = 10

def build_label_vocab(series, min_freq=MIN_CATEGORY_FREQ):
    counts = series.value_counts()
    frequent = sorted(counts[counts >= min_freq].index.tolist())
    return {c: i + 1 for i, c in enumerate(frequent)}  # 0 = 희귀/미확인 통합 버킷

def encode_labels(series, vocab):
    return series.map(vocab).fillna(0).astype(np.int64).values

brand_vocab = build_label_vocab(train_df['brand_name'])
cat_main_vocab = build_label_vocab(train_df['cat_main'])
cat_sub_vocab = build_label_vocab(train_df['cat_sub'])
cat_sub2_vocab = build_label_vocab(train_df['cat_sub2'])

print(f'브랜드 : 전체 {train_df["brand_name"].nunique()}종 -> 빈도 {MIN_CATEGORY_FREQ}회 이상만 개별 임베딩 {len(brand_vocab)}종 (나머지는 희귀 버킷)')
print(f'cat_main  : {train_df["cat_main"].nunique()}종 -> {len(cat_main_vocab)}종')
print(f'cat_sub   : {train_df["cat_sub"].nunique()}종 -> {len(cat_sub_vocab)}종')
print(f'cat_sub2  : {train_df["cat_sub2"].nunique()}종 -> {len(cat_sub2_vocab)}종')

brand_train_ids = encode_labels(train_df['brand_name'], brand_vocab)
brand_val_ids = encode_labels(val_df['brand_name'], brand_vocab)
brand_test_ids = encode_labels(test_df['brand_name'], brand_vocab)
cat_main_train_ids = encode_labels(train_df['cat_main'], cat_main_vocab)
cat_main_val_ids = encode_labels(val_df['cat_main'], cat_main_vocab)
cat_main_test_ids = encode_labels(test_df['cat_main'], cat_main_vocab)
cat_sub_train_ids = encode_labels(train_df['cat_sub'], cat_sub_vocab)
cat_sub_val_ids = encode_labels(val_df['cat_sub'], cat_sub_vocab)
cat_sub_test_ids = encode_labels(test_df['cat_sub'], cat_sub_vocab)
cat_sub2_train_ids = encode_labels(train_df['cat_sub2'], cat_sub2_vocab)
cat_sub2_val_ids = encode_labels(val_df['cat_sub2'], cat_sub2_vocab)
cat_sub2_test_ids = encode_labels(test_df['cat_sub2'], cat_sub2_vocab)

rare_ratio_train = (brand_train_ids == 0).mean()
print(f'\ntrain에서 브랜드가 "희귀" 버킷(0번)으로 들어간 비율 : {rare_ratio_train:.1%}')

In [ ]:
# 수치형 피처 스케일링 - train_df로만 fit, val/test는 transform만
numeric_cols = ['item_condition_id', 'has_description', 'name_len', 'desc_len']
scaler = StandardScaler()
X_num_train = scaler.fit_transform(train_df[numeric_cols]).astype(np.float32)
X_num_val = scaler.transform(val_df[numeric_cols]).astype(np.float32)
X_num_test = scaler.transform(test_df[numeric_cols]).astype(np.float32)
print('수치형 스케일링 shape :', X_num_train.shape, X_num_val.shape, X_num_test.shape)

# 구간 G. 메인 가격예측 모델
DataLoader 구성부터 최종 Ridge vs MLP 비교까지.

In [ ]:
# PyTorch Dataset/DataLoader 구성
class MercariDataset(Dataset):
    def __init__(self, brand_ids, cat_main_ids, cat_sub_ids, cat_sub2_ids,
                 name_ids, desc_ids, num_feats, shipping, y):
        self.brand_ids = torch.as_tensor(brand_ids, dtype=torch.long)
        self.cat_main_ids = torch.as_tensor(cat_main_ids, dtype=torch.long)
        self.cat_sub_ids = torch.as_tensor(cat_sub_ids, dtype=torch.long)
        self.cat_sub2_ids = torch.as_tensor(cat_sub2_ids, dtype=torch.long)
        self.name_ids = torch.as_tensor(name_ids, dtype=torch.long)
        self.desc_ids = torch.as_tensor(desc_ids, dtype=torch.long)
        self.num_feats = torch.as_tensor(num_feats, dtype=torch.float32)
        self.shipping = torch.as_tensor(shipping.values.astype(np.float32), dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.brand_ids)

    def __getitem__(self, idx):
        x = (
            self.brand_ids[idx], self.cat_main_ids[idx], self.cat_sub_ids[idx], self.cat_sub2_ids[idx],
            self.name_ids[idx], self.desc_ids[idx], self.num_feats[idx], self.shipping[idx],
        )
        return x, self.y[idx]

def make_dataset(df, brand_ids, cat_main_ids, cat_sub_ids, cat_sub2_ids, name_ids, desc_ids, num_feats):
    return MercariDataset(brand_ids, cat_main_ids, cat_sub_ids, cat_sub2_ids,
                           name_ids, desc_ids, num_feats, df['shipping'], df['price'].values)

train_dataset = make_dataset(train_df, brand_train_ids, cat_main_train_ids, cat_sub_train_ids, cat_sub2_train_ids,
                              name_train_ids, desc_train_ids, X_num_train)
val_dataset = make_dataset(val_df, brand_val_ids, cat_main_val_ids, cat_sub_val_ids, cat_sub2_val_ids,
                            name_val_ids, desc_val_ids, X_num_val)
test_dataset = make_dataset(test_df, brand_test_ids, cat_main_test_ids, cat_sub_test_ids, cat_sub2_test_ids,
                             name_test_ids, desc_test_ids, X_num_test)

BATCH_SIZE = 1024
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print('train batch 수 :', len(train_loader), ' val batch 수 :', len(val_loader), ' test batch 수 :', len(test_loader))

In [ ]:
# 모델 정의 - 임베딩 + MLP(비선형)
class MercariMLP(nn.Module):
    def __init__(self, n_brand, n_cat_main, n_cat_sub, n_cat_sub2,
                 name_vocab_size, desc_vocab_size, n_numeric=4,
                 brand_dim=16, cat_dim=8, word_dim=32, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.brand_emb = nn.Embedding(n_brand + 1, brand_dim, padding_idx=0)
        self.cat_main_emb = nn.Embedding(n_cat_main + 1, cat_dim, padding_idx=0)
        self.cat_sub_emb = nn.Embedding(n_cat_sub + 1, cat_dim, padding_idx=0)
        self.cat_sub2_emb = nn.Embedding(n_cat_sub2 + 1, cat_dim, padding_idx=0)
        self.name_emb = nn.EmbeddingBag(name_vocab_size + 2, word_dim, mode='mean', padding_idx=0)
        self.desc_emb = nn.EmbeddingBag(desc_vocab_size + 2, word_dim, mode='mean', padding_idx=0)

        concat_dim = brand_dim + cat_dim * 3 + word_dim * 2 + n_numeric + 1
        self.mlp = nn.Sequential(
            nn.Linear(concat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, brand_ids, cat_main_ids, cat_sub_ids, cat_sub2_ids, name_ids, desc_ids, num_feats, shipping):
        parts = [
            self.brand_emb(brand_ids),
            self.cat_main_emb(cat_main_ids),
            self.cat_sub_emb(cat_sub_ids),
            self.cat_sub2_emb(cat_sub2_ids),
            self.name_emb(name_ids),
            self.desc_emb(desc_ids),
            num_feats,
            shipping.unsqueeze(1),
        ]
        x = torch.cat(parts, dim=1)
        return self.mlp(x).squeeze(1)

model = MercariMLP(
    n_brand=len(brand_vocab), n_cat_main=len(cat_main_vocab),
    n_cat_sub=len(cat_sub_vocab), n_cat_sub2=len(cat_sub2_vocab),
    name_vocab_size=len(name_vocab), desc_vocab_size=len(desc_vocab),
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'전체 파라미터 수 : {n_params:,}')

In [ ]:
# 학습 - MSE loss (= RMSLE, log1p 스케일). val RMSLE 기준 조기종료.
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, n_samples = 0.0, 0
    criterion = nn.MSELoss()

    with torch.set_grad_enabled(is_train):
        for (brand_ids, cat_main_ids, cat_sub_ids, cat_sub2_ids, name_ids, desc_ids, num_feats, shipping), y in loader:
            brand_ids, cat_main_ids = brand_ids.to(device), cat_main_ids.to(device)
            cat_sub_ids, cat_sub2_ids = cat_sub_ids.to(device), cat_sub2_ids.to(device)
            name_ids, desc_ids = name_ids.to(device), desc_ids.to(device)
            num_feats, shipping, y = num_feats.to(device), shipping.to(device), y.to(device)

            pred = model(brand_ids, cat_main_ids, cat_sub_ids, cat_sub2_ids, name_ids, desc_ids, num_feats, shipping)
            loss = criterion(pred, y)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(y)
            n_samples += len(y)

    return total_loss / n_samples

N_EPOCHS = 30
PATIENCE = 3
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_val_rmsle = float('inf')
best_state = None
epochs_without_improve = 0
history = []

for epoch in range(1, N_EPOCHS + 1):
    train_mse = run_epoch(model, train_loader, optimizer)
    val_mse = run_epoch(model, val_loader, optimizer=None)
    train_rmsle, val_rmsle = np.sqrt(train_mse), np.sqrt(val_mse)
    history.append({'epoch': epoch, 'train_rmsle': train_rmsle, 'val_rmsle': val_rmsle})
    print(f'epoch {epoch:2d}/{N_EPOCHS}  train RMSLE={train_rmsle:.5f}  val RMSLE={val_rmsle:.5f}', end='')

    if val_rmsle < best_val_rmsle:
        best_val_rmsle = val_rmsle
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improve = 0
        print('  <- best')
    else:
        epochs_without_improve += 1
        print(f'  (연속 {epochs_without_improve}회 개선 없음)')
        if epochs_without_improve >= PATIENCE:
            print(f'\nval RMSLE가 {PATIENCE}epoch 연속 개선되지 않아 조기 종료합니다.')
            break

model.load_state_dict(best_state)
history_df = pd.DataFrame(history)
print(f'\n최적 val RMSLE : {best_val_rmsle:.5f} (epoch {history_df.loc[history_df["val_rmsle"].idxmin(), "epoch"]:.0f})')

In [ ]:
# 학습 곡선
plt.figure(figsize=(6, 4))
plt.plot(history_df['epoch'], history_df['train_rmsle'], marker='o', color='#2a78d6', label='train')
plt.plot(history_df['epoch'], history_df['val_rmsle'], marker='o', color='#eb6834', label='val')
best_epoch = history_df.loc[history_df['val_rmsle'].idxmin(), 'epoch']
plt.axvline(best_epoch, color='#26a88f', linestyle='--', label=f'최적 epoch ({best_epoch:.0f})')
plt.xlabel('epoch')
plt.ylabel('RMSLE (lower is better)')
plt.title('MLP 학습 곡선 (train vs val)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 최종 평가 - test_df는 여기서 "딱 한 번" 등장한다.
def predict_log(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for (brand_ids, cat_main_ids, cat_sub_ids, cat_sub2_ids, name_ids, desc_ids, num_feats, shipping), y in loader:
            brand_ids, cat_main_ids = brand_ids.to(device), cat_main_ids.to(device)
            cat_sub_ids, cat_sub2_ids = cat_sub_ids.to(device), cat_sub2_ids.to(device)
            name_ids, desc_ids = name_ids.to(device), desc_ids.to(device)
            num_feats, shipping = num_feats.to(device), shipping.to(device)
            pred = model(brand_ids, cat_main_ids, cat_sub_ids, cat_sub2_ids, name_ids, desc_ids, num_feats, shipping)
            preds.append(pred.cpu().numpy())
    return np.concatenate(preds)

def rmsle_on_log_scale(y_true_log, y_pred_log):
    return np.sqrt(mean_squared_error(y_true_log, y_pred_log))

pred_log = predict_log(model, test_loader)
test_rmsle = rmsle_on_log_scale(test_df['price'].values, pred_log)

print(f'[MLP] 최종 test RMSLE : {test_rmsle:.5f}   (val 기준 최적 epoch {best_epoch:.0f}의 가중치로 평가)')
print(f'[참고] 릿지 최종 RMSLE(alpha=3.1623, test) : 0.45223')

In [ ]:
# 실제 가격 vs 예측 가격 비교
actual_price = np.expm1(test_df['price'].values)
predicted_price = np.expm1(pred_log)

price_compare = pd.DataFrame({
    '상품명': test_df['name'],
    '실제 가격': actual_price.round(2),
    '예측 가격': predicted_price.round(2),
})
price_compare['차이(예측-실제)'] = (price_compare['예측 가격'] - price_compare['실제 가격']).round(2)
price_compare['오차율(%)'] = (price_compare['차이(예측-실제)'] / price_compare['실제 가격'] * 100).round(1)

print(f'평균 절대 오차(MAE) : ${price_compare["차이(예측-실제)"].abs().mean():.2f}')
print(f'중앙값 절대 오차     : ${price_compare["차이(예측-실제)"].abs().median():.2f}')
print(f'평균 오차(부호 포함) : ${price_compare["차이(예측-실제)"].mean():+.2f}  (양수면 평균적으로 비싸게 예측)')

In [ ]:
# 결과 확인 - 무작위 표본 15개 (실제 가격 순 정렬)
price_compare.sample(15, random_state=RANDOM_STATE).sort_values('실제 가격')

In [ ]:
# 실제 vs 예측 가격 - 전체 테스트셋을 그림으로 확인
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(actual_price, predicted_price, alpha=0.05, s=8, color='#2a78d6')
max_val = 200
axes[0].plot([0, max_val], [0, max_val], color='#eb6834', linewidth=1.5, label='y = x (완벽한 예측)')
axes[0].set_xlim(0, max_val)
axes[0].set_ylim(0, max_val)
axes[0].set_xlabel('실제 가격 ($)')
axes[0].set_ylabel('예측 가격 ($)')
axes[0].set_title('MLP: 실제 vs 예측 (산점도)')
axes[0].legend()

sns.histplot(price_compare['차이(예측-실제)'], bins=100, ax=axes[1], color='#2a78d6')
axes[1].set_xlim(-100, 100)
axes[1].axvline(0, color='#eb6834', linestyle='--', label='오차 0 (정확히 맞춘 지점)')
axes[1].set_xlabel('예측 오차 ($) = 예측 가격 - 실제 가격')
axes[1].set_title('MLP: 오차 분포')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 릿지 vs MLP 최종 비교
comparison = pd.DataFrame({
    '모델': ['Ridge (선형, alpha=3.1623)', f'MLP (임베딩+비선형, val기준 최적 epoch {best_epoch:.0f})'],
    'RMSLE': [0.45223, test_rmsle],
})

In [ ]:
# 결과 확인
comparison

In [ ]:
diff = comparison['RMSLE'][1] - comparison['RMSLE'][0]
better = 'MLP' if diff < 0 else 'Ridge'
print(f'차이: {diff:+.5f}  ({better}가 더 좋음)')

plt.figure(figsize=(5, 4))
bars = plt.bar(comparison['모델'], comparison['RMSLE'], color=['#c3c2b7', '#2a78d6'])
plt.ylabel('RMSLE (lower is better)')
plt.title('Ridge vs MLP')
plt.xticks(rotation=10, ha='right')
for b in bars:
    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.002, f'{b.get_height():.5f}', ha='center')
plt.ylim(0, max(comparison['RMSLE']) * 1.15)
plt.tight_layout()
plt.show()